# version of the MoivieSimilarities but with dataframes instead

In [ ]:
# save to parquette, change the RDD into a dataframe

import sys
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.functions import udf
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType
from math import sqrt


def computeCosineSimilarity(spark, data):
    # Compute xx, xy and yy columns
    pairScores = data \
      .withColumn("xx", func.col("rating1") * func.col("rating1")) \
      .withColumn("yy", func.col("rating2") * func.col("rating2")) \
      .withColumn("xy", func.col("rating1") * func.col("rating2")) 

    # Compute numerator, denominator and numPairs columns
    calculateSimilarity = pairScores \
      .groupBy("movie1", "movie2") \
      .agg( \
        func.sum(func.col("xy")).alias("numerator"), \
        (func.sqrt(func.sum(func.col("xx"))) * func.sqrt(func.sum(func.col("yy")))).alias("denominator"), \
        func.count(func.col("xy")).alias("numPairs")
      )

    # Calculate score and select only needed columns (movie1, movie2, score, numPairs)
    result = calculateSimilarity \
      .withColumn("score", \
        func.when(func.col("denominator") != 0, func.col("numerator") / func.col("denominator")) \
          .otherwise(0) \
      ).select("movie1", "movie2", "score", "numPairs")

    return result



# ----------------------------
# Main Spark setup
# ----------------------------

# creating spark session
spark = SparkSession.builder.appName('ml-1m').getOrCreate()

spark.sparkContext.setLogLevel("FATAL")

# ----------------------------
# S3 paths
# ----------------------------
MOVIES_PATH = "s3a://link2-bucket-rev-005311909391-us-east-2-an/ml-1m/movies.dat"
RATINGS_PATH = "s3a://link2-bucket-rev-005311909391-us-east-2-an/ml-1m/ratings.dat"


# ----------------------------
# Load and broadcast movie names
# ----------------------------
print("Loading movie names from S3...")

movieNamesSchema = StructType([
    StructField("movieId", IntegerType()),
    StructField("movie_name", StringType())
])

# making a df out of the movieNames
movieNames = spark.read \
      .option("sep", "::") \
      .option("charset", "ISO-8859-1") \
      .schema(movieNamesSchema) \
      .csv(MOVIES_PATH)


# ----------------------------
# Load ratings from S3
# ----------------------------
print("Loading ratings from S3...")

schema = StructType([
    StructField("userId", IntegerType()),
    StructField('movieId', IntegerType()),
    StructField("rating", FloatType())
])

movies = spark.read.option("sep", "::").csv(RATINGS_PATH, schema=schema)

ratings = movies.select(func.col('userId'), func.col('movieId'), func.col("rating"))

# ----------------------------
# Build movie pairs
# ----------------------------

ratingsPartitioned = ratings.repartition(func.col('userId'))


# Emit every movie rated together by the same user.
# Self-join to find every combination.
# Select movie pairs and rating pairs
moviePairs = ratingsPartitioned.alias("ratings1") \
      .join(ratingsPartitioned.alias("ratings2"), (func.col("ratings1.userId") == func.col("ratings2.userId")) \
            & (func.col("ratings1.movieId") < func.col("ratings2.movieId"))) \
      .select(func.col("ratings1.movieId").alias("movie1"), \
        func.col("ratings2.movieId").alias("movie2"), \
        func.col("ratings1.rating").alias("rating1"), \
        func.col("ratings2.rating").alias("rating2"))


moviePairSimilarities = computeCosineSimilarity(spark, moviePairs).persist()



# save full results
moviePairSimilarities.write.mode("overwrite").parquet("s3a://link2-bucket-rev-005311909391-us-east-2-an/output/movie-sims")

# ----------------------------
# Query similar movies
# ----------------------------
if len(sys.argv) > 1:

  # grabs the movie that the user inputs
  movieID = int(sys.argv[1])

  scoreThreshold = 0.97
  coOccurrenceThreshold = 50

  # Filter for movies with this sim that are "good" as defined by
  # our quality thresholds above
  filteredResults = moviePairSimilarities.filter( \
      ((func.col("movie1") == movieID) | (func.col("movie2") == movieID)) & \
        (func.col("score") > scoreThreshold) & (func.col("numPairs") > coOccurrenceThreshold))
    



  # getting the top 10 scores
  results = filteredResults.orderBy(func.col("score"), ascending=False).limit(10)

  # adding the movieid to each of the movies in the filtered results
  # when the id is equal to movie1, have the movieId be movie2 otherwise have it be movie1
  results = results.withColumn(
      "movieId",
      func.when(
          func.col("movie1") == movieID,
          func.col("movie2")
      ).otherwise(func.col("movie1"))
  )


  # broadcast join to get the name of the movie id. largeDf.join(broadcast(smallDf))
  join_results = movieNames.join(func.broadcast(results),"movieId")

  # getting the name using the movieId from the movieNames df
  name = movieNames.filter(func.col('movieId') == movieID).select(func.col("movie_name")).first()


  print("\nTop 10 similar movies for:",
        name.movie_name)

  # printing out all the movies that ended up in the join_results table
  for result in join_results.collect():
      print(
          result.movie_name,
          "\tscore:", result.score,
          "\tstrength:", result.numPairs
      )

# getting information from the customers.csv and the orders.csv to only return customers with no orders

In [5]:
# use customers.cs, orders.csv to get customers with no orders

from pyspark.sql import SparkSession, functions as func

spark = SparkSession.builder.appName("orders").getOrCreate()

# get the dataframes for both the csvs
customers = spark.read.option("sep", ",").csv("../../customers.csv", header=True, inferSchema=True)
orders = spark.read.option("sep", ",").csv("../../orders.csv", header=True, inferSchema=True)

# do an outer join to get see all of the connections
outer_join = customers.join(orders, "customer_id", "outer")


no_orders = outer_join.filter(func.col('order_id').isNull())

# show the final graph
final = no_orders.select(func.col('first_name'), func.col('last_name'), func.col('order_id').alias('total_orders_made')).show()

+----------+---------+-----------------+
|first_name|last_name|total_orders_made|
+----------+---------+-----------------+
|       Bob|    Smith|             NULL|
|     David|   Wilson|             NULL|
|     Grace|    Moore|             NULL|
|       Ivy| Anderson|             NULL|
|       Mia|   Martin|             NULL|
|    Olivia|   Garcia|             NULL|
+----------+---------+-----------------+

